In [ ]:
import os
import logging

import duckdb
import polars as pl

from ml_deduplication.modeling.xgboost.preprocessing import (
    preprocess_entities_df,
    preprocess_entities_df_batched,
    preprocess_features
)
from pathlib import Path
from typing import Any


In [ ]:
logging.basicConfig()
logger = logging.getLogger()

In [ ]:
df_acteurs = pl.read_parquet("../../datasets/acteurs_commerces.parquet")
df_embeddings = pl.read_parquet("../../datasets/acteurs_commerces_adresses_embeddings.parquet")

In [ ]:
df_features_preprocessed = preprocess_features(df_acteurs,df_embeddings=df_embeddings)

In [ ]:
df_features_preprocessed


In [ ]:
from typing import Generator
def create_duckdb_con(database_file:Path|str=":memory:",temp_dir : Path | None=None, memory_limit:str|None =None) -> duckdb.DuckDBPyConnection:

    config: dict[Any,Any] = {}
    if temp_dir is not None:
        os.makedirs(temp_dir, exist_ok=True)
        config["temp_directory"]= temp_dir
    if memory_limit is not None:
        config["memory_limit"] = memory_limit
    con = duckdb.connect(database=database_file,config=config)
    con.install_extension("spatial")
    con.load_extension("spatial")

    return con

def load_df_acteurs_to_duckdb(df_acteurs: pl.DataFrame, con: duckdb.DuckDBPyConnection,additional_columns_to_keep: list[str] | None = None,):
    con.register("acteurs", df_acteurs.with_columns([
                pl.col("latitude").cast(pl.Float32),
                pl.col("longitude").cast(pl.Float32),
            ]))
    
    cols_needed = [
            "identifiant_unique",
            "siren",
            "code_postal",
            "latitude",
            "longitude",
            "source_id",
            "acteur_type_id",
        ]
    if additional_columns_to_keep is not None:
        for colname in additional_columns_to_keep:
            if colname not in cols_needed:
                cols_needed.append(colname)

    cols_needed_expr = ",".join(cols_needed)
    con.sql(f"""
    CREATE TABLE features AS
        SELECT 
            *,
            ST_Point2D(longitude,latitude) as location
        FROM acteurs;
    CREATE TABLE features_minimal AS
        SELECT 
            {cols_needed_expr},
            ST_Point2D(longitude,latitude) as location
        FROM acteurs;
    """)

def run_blocking(con: duckdb.DuckDBPyConnection, additional_business_rules_sql_exprs: list[str] | None = None,):
    business_rules_filter_sql = """
        (
            (coalesce(acteur_type_id_l,-1)=coalesce(acteur_type_id_r,-1))
            OR (
                acteur_type_id_l=4 AND acteur_type_id_r=3
            )
            OR (
                acteur_type_id_l=3 AND acteur_type_id_r=4
            )
        )
        AND (
            coalesce(source_id_l,-1) <> coalesce(source_id_r,-2)
        )
    """
    if additional_business_rules_sql_exprs is not None:
        for additional_rule in additional_business_rules_sql_exprs:
            business_rules_filter_sql += f"AND ({additional_rule})"
    # =====================================================================
    # ÉTAPE 1 : Génération de candidats sur le schéma minimal
    # =====================================================================

    

    # 1. Code Postal
    con.sql(f"""
    CREATE table blocking_departement AS (
        with joined as (
            SELECT
                columns(l.*) as '\\0_l',
                columns(r.*) as '\\0_r',
            from
                features_minimal l
            inner join features_minimal r on
                l.code_postal[:2]=r.code_postal[:2]
        )
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from joined
        where identifiant_unique_l < identifiant_unique_r
        AND {business_rules_filter_sql}
    )
    """)
    logger.info(
        "Blocking: cp predicate -> %s candidates ",
        con.sql("SELECT count(*) FROM blocking_departement").limit(1).fetchall()[0][0],
    )

    # 2. Grille Géographique optimisée en jointure grace à ST_DWITHIN (distance en degré, correspond à 15km)
    con.sql("""
    CREATE table blocking_geo_temp AS (
        SELECT
            columns(l.*) as '\\0_l',
            columns(r.*) as '\\0_r'
        from
            features_minimal l
        inner join features_minimal r on
            ST_DWithin(
                l.location,
                r.location,
                0.14
            )
    )
    """)

    con.sql(f"""
    CREATE table blocking_geo AS (
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from blocking_geo_temp
        anti join blocking_departement using (identifiant_unique_l, identifiant_unique_r)
        where identifiant_unique_l < identifiant_unique_r
        AND {business_rules_filter_sql}
    )
    """)

    logger.info(
        "Blocking: geo predicate -> %s candidates",
        con.sql("SELECT count(*) FROM blocking_geo").limit(1).fetchall()[0][0],
    )

    # 3. SIREN
    con.sql(f"""
    CREATE table blocking_siren AS (
        with joined as (
            SELECT
                columns(l.*) as '\\0_l',
                columns(r.*) as '\\0_r',
            from
                features_minimal l
            inner join features_minimal r on
                l.siren=r.siren
        )
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from joined
        anti join blocking_departement using (identifiant_unique_l, identifiant_unique_r)
        anti join blocking_geo using (identifiant_unique_l, identifiant_unique_r)
        where identifiant_unique_l < identifiant_unique_r
        AND {business_rules_filter_sql}
        
    )
    """)

    logger.info(
        "Blocking: siren predicate -> %s candidates ",
        con.sql("SELECT count(*) FROM blocking_siren").limit(1).fetchall()[0][0],
    )

    

    # =====================================================================
    # ÉTAPE 3 : Union + dédoublonnage par buckets (mémoire bornée)
    # =====================================================================
    # Les trois prédicats ont été réduits AVANT dédoublonnage (distance + règles
    # métier), donc la concaténation ne fait pas exploser la mémoire. Le
    # dédoublonnage est ensuite partitionné en `dedup_buckets` buckets via un
    # hash déterministe de la paire (l, r) : chaque `unique()` n'opère que sur
    # un bucket, donc la table de hachage est bornée par la taille du plus gros
    # bucket — pas par l'ensemble complet. C'est ce qui supprime le pic ~200GB.
    logger.info("Collecting minimal valid pairs to free memory...")
    con.sql("""
        CREATE TABLE pairs AS
        SELECT identifiant_unique_l, identifiant_unique_r, 'siren'      AS blocking_origin FROM blocking_siren
        UNION ALL
        SELECT identifiant_unique_l, identifiant_unique_r, 'departement' AS blocking_origin FROM blocking_departement
        UNION ALL
        SELECT identifiant_unique_l, identifiant_unique_r, 'geo'         AS blocking_origin FROM blocking_geo
        """)

    logger.info(
        "Blocking: dedup complete -> %s valid candidate pairs ",
        con.sql("SELECT count(*) FROM pairs").limit(1).fetchall()[0][0],
    )
    con.sql("CHECKPOINT;")
    for table in [
            "blocking_siren",
            "blocking_departement",
            "blocking_geo",
            "blocking_geo_temp"
    ]:
        con.sql(f"DROP TABLE IF EXISTS {table}")
    con.sql("CHECKPOINT;")

def generate_features(con: duckdb.DuckDBPyConnection,additional_columns_to_keep: list[str] | None = None):
    feature_exprs = [
            (
                "nom_clean_dist",
                "1 - jaro_winkler_similarity(f.nom_clean, f2.nom_clean)",
            ),
            (
                "ville_clean_dist",
                "1 - jaro_winkler_similarity(f.ville_clean, f2.ville_clean)",
            ),
            (
                "adresse_clean_distance",
                (
                    "coalesce(CASE "
                    "WHEN f.adresse_clean_vector IS NULL OR f2.adresse_clean_vector IS NULL "
                    "THEN NULL "
                    "ELSE 1 - list_cosine_similarity("
                    "f.adresse_clean_vector, f2.adresse_clean_vector) "
                    "END, 'NaN'::REAL)"
                ),
            ),
            ("siren_match", "f.siren = f2.siren"),
            ("siret_match", "f.siret = f2.siret"),
            ("telephone_match", "f.telephone = f2.telephone"),
            ("code_commune_insee_match", "f.code_commune_insee = f2.code_commune_insee"),
            ("code_postal_match", "f.code_postal = f2.code_postal"),
            (
                "departement_match",
                "substr(f.code_postal, 1, 2) = substr(f2.code_postal, 1, 2)",
            ),
            ("acteur_type_id_l", "f.acteur_type_id"),
            ("acteur_type_id_r", "f2.acteur_type_id"),
            ("geo_distance","ST_DISTANCE_Spheroid(ST_Point2D(f.latitude,f.longitude),ST_Point2D(f2.latitude,f2.longitude))")
        ]

    # Colonnes `_l/_r` supplémentaires demandées (source_id, parent_id, colonnes
    # de linkage…). acteur_type_id est déjà une feature, on l'exclut.
    feature_l_names = {name for name, _ in feature_exprs}
    additional_pair_exprs = []
    if additional_columns_to_keep is not None:
        for col in additional_columns_to_keep:
            if f"{col}_l" not in feature_l_names:
                additional_pair_exprs.append(f"f.{col} AS {col}_l")
                additional_pair_exprs.append(f"f2.{col} AS {col}_r")

    # Column order must mirror the polars `generate_features` output so the two
    # paths are interchangeable (batch concatenation equals the full pipeline).
    ordered_features = [
        "nom_clean_dist",
        "adresse_clean_distance",
        "ville_clean_dist",
        "siren_match",
        "siret_match",
        "telephone_match",
        "code_commune_insee_match",
        "code_postal_match",
        "departement_match",
        "geo_distance",
        "acteur_type_id_l",
        "acteur_type_id_r",
    ]
    expr_by_name = {name: expr for name, expr in feature_exprs}
    ordered_select = []
    for name in ordered_features:
        ordered_select.append(f"    {expr_by_name[name]} AS {name},")
    select_lines = "\n".join(
        [
            "    p.identifiant_unique_l,",
            "    p.identifiant_unique_r,",
            *ordered_select,
            *[f"    {expr}," for expr in additional_pair_exprs],
        ]
    ).rstrip(",")
    
    con.sql(f"""
        SET geometry_always_xy = true;
        CREATE TABLE pairs_features AS
        (
            SELECT
            {select_lines}
            FROM pairs p
            left join features f on p.identifiant_unique_l=f.identifiant_unique
            left join features f2 on p.identifiant_unique_r=f2.identifiant_unique
        )
    """)
    
    #for table in ["pairs","features"]:
    #    con.sql(f"DROP TABLE IF EXISTS {table}")
    con.sql("CHECKPOINT;")

def get_features_df_from_duckdb(con: duckdb.DuckDBPyConnection, batch_size:int| None = None) -> pl.DataFrame | Generator[pl.DataFrame | pl.Series]:
    
    sql = "SELECT * FROM pairs_features"
    if batch_size is not None:
        reader = con.sql(
                    sql
                ).to_arrow_reader(batch_size=batch_size)
        for batch_arrow in reader:
            yield pl.from_arrow(batch_arrow)  
    else:
        return con.sql(sql).pl()


In [ ]:
con = create_duckdb_con(temp_dir=Path("/Volumes/PRO-G40/duckdb_tmp"))

In [ ]:
load_df_acteurs_to_duckdb(con=con,df_acteurs=df_features_preprocessed)

In [ ]:
business_rules_filter_sql = """
        (
            (coalesce(acteur_type_id_l,-1)=coalesce(acteur_type_id_r,-1))
            OR (
                acteur_type_id_l=4 AND acteur_type_id_r=3
            )
            OR (
                acteur_type_id_l=3 AND acteur_type_id_r=4
            )
        )
        AND (
            coalesce(source_id_l,-1) <> coalesce(source_id_r,-2)
        )
    """
print(con.sql(f"""
EXPLAIN with joined as (
            SELECT
                columns(l.*) as '\\0_l',
                columns(r.*) as '\\0_r',
            from
                features l
            inner join features r on
                l.code_postal[:2]=r.code_postal[:2]
            where identifiant_unique_l < identifiant_unique_r
                    AND {business_rules_filter_sql}
        )
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from joined
""").fetchall()[0][1])


In [ ]:
run_blocking(con=con)

In [ ]:
generate_features(con=con)

In [ ]:
features_df_generator = get_features_df_from_duckdb(con=con,batch_size=10_0000)


In [ ]:
for df in features_df_generator:
    print(len(df))
    break

In [ ]:
def _prepare_pairs_final_table(
    df_features: pl.DataFrame,
    additional_business_rules_sql_exprs: list[str] | None = None,
    additional_columns_to_keep: list[str] | None = None,
) -> tuple[duckdb.DuckDBPyConnection | None, int]:
    """Run the duckdb blocking pipeline and leave candidate pairs ready to join.

    Blocks candidate pairs (siren / departement / geo predicates, deduplicated
    by hashed buckets) into the ``pairs_minimal`` table and registers the
    feature tables needed by the final pair/feature joins. Intermediate
    blocking tables are dropped to free the duckdb baseline.

    Args:
        df_features: Preprocessed entities frame (output of `preprocess_features`).
        additional_business_rules_sql_exprs: Extra SQL predicates ANDed into the
            blocking filter.
        additional_columns_to_keep: Entity columns to carry through as ``_l``/``_r``
            in the output.

    Returns:
        A tuple ``(con, n_pairs)``. If no candidate pairs were found, ``con`` is
        ``None`` (connection already closed) and ``n_pairs`` is ``0``. Otherwise
        ``con`` holds ``pairs_minimal``, ``features_full`` and ``features_vector``
        and is left open for the caller to build its final table and stream over.
    """
    os.makedirs(DUCKDB_TEMP_DIR, exist_ok=True)
    config = {"temp_directory": DUCKDB_TEMP_DIR}
    if DUCKDB_MEMORY_LIMIT is not None:
        config["memory_limit"] = DUCKDB_MEMORY_LIMIT
    con = duckdb.connect(config=config)
    con.install_extension("spatial")
    con.load_extension("spatial")

    cols_needed = [
        "identifiant_unique",
        "siren",
        "code_postal",
        "latitude",
        "longitude",
        "source_id",
        "acteur_type_id",
    ]
    if additional_columns_to_keep is not None:
        for colname in additional_columns_to_keep:
            if colname not in cols_needed:
                cols_needed.append(colname)

    business_rules_filter_sql = """
    (
        (coalesce(acteur_type_id_l,-1)=coalesce(acteur_type_id_r,-1))
        OR (
            acteur_type_id_l=4 AND acteur_type_id_r=3
        )
        OR (
            acteur_type_id_l=3 AND acteur_type_id_r=4
        )
    )
    AND (
        coalesce(source_id_l,-1) <> coalesce(source_id_r,-2)
    )
"""
    if additional_business_rules_sql_exprs is not None:
        for additional_rule in additional_business_rules_sql_exprs:
            business_rules_filter_sql += f"AND ({additional_rule})"

    business_rules_filter_sql = f"({business_rules_filter_sql})"

    # Optimize types immediately and register the minimal schema for blocking.
    df_minimal = df_features.select(cols_needed).with_columns(
        [
            pl.col("latitude").cast(pl.Float32),
            pl.col("longitude").cast(pl.Float32),
        ]
    )
    con.register("df_minimal", df_minimal)
    con.sql("""CREATE TABLE features AS
        SELECT *,
               ST_Point2D(longitude,latitude) as location
        FROM df_minimal
    """)

    # =====================================================================
    # ÉTAPE 1 : Génération de candidats sur le schéma minimal
    # =====================================================================

    # 1. SIREN
    con.sql(f"""
    CREATE table blocking_siren AS (
        with joined as (
            SELECT
                columns(l.*) as '\\0_l',
                columns(r.*) as '\\0_r',
            from
                features l
            inner join features r on
                l.siren=r.siren
        )
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from joined
        where identifiant_unique_l < identifiant_unique_r
        AND {business_rules_filter_sql}
    )
""")

    logger.info(
        "Blocking: siren predicate -> %s candidates ",
        con.sql("SELECT count(*) FROM blocking_siren").limit(1).fetchall()[0][0],
    )

    # 2. Code Postal
    con.sql(f"""
    CREATE table blocking_departement AS (
        with joined as (
            SELECT
                columns(l.*) as '\\0_l',
                columns(r.*) as '\\0_r',
            from
                features l
            inner join features r on
                l.code_postal[:2]=r.code_postal[:2]
        )
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from joined
        where identifiant_unique_l < identifiant_unique_r
        AND {business_rules_filter_sql}
    )
    """)
    logger.info(
        "Blocking: cp predicate -> %s candidates ",
        con.sql("SELECT count(*) FROM blocking_departement").limit(1).fetchall()[0][0],
    )

    # 3. Grille Géographique optimisée en jointure grace à ST_DWITHIN (distance en degré, correspond à 15km)
    con.sql("""
    CREATE table blocking_geo_temp AS (
        SELECT
            columns(l.*) as '\\0_l',
            columns(r.*) as '\\0_r',
        from
            features l
        inner join features r on
            ST_DWithin(
                l.location,
                r.location,
                0.14
            )
    )
    """)

    con.sql(f"""
    CREATE table blocking_geo AS (
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from blocking_geo_temp
        where identifiant_unique_l < identifiant_unique_r
        AND {business_rules_filter_sql}
    )
    """)

    logger.info(
        "Blocking: geo predicate -> %s candidates",
        con.sql("SELECT count(*) FROM blocking_geo").limit(1).fetchall()[0][0],
    )

    # =====================================================================
    # ÉTAPE 3 : Union + dédoublonnage par buckets (mémoire bornée)
    # =====================================================================
    # Les trois prédicats ont été réduits AVANT dédoublonnage (distance + règles
    # métier), donc la concaténation ne fait pas exploser la mémoire. Le
    # dédoublonnage est ensuite partitionné en `dedup_buckets` buckets via un
    # hash déterministe de la paire (l, r) : chaque `unique()` n'opère que sur
    # un bucket, donc la table de hachage est bornée par la taille du plus gros
    # bucket — pas par l'ensemble complet. C'est ce qui supprime le pic ~200GB.
    logger.info("Collecting minimal valid pairs to free memory...")
    con.sql("""
        CREATE table pairs AS (
        with all_data as (
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from blocking_siren
        UNION ALL
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from blocking_departement
        UNION ALL
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from blocking_geo
        )
        SELECT
            identifiant_unique_l,
            identifiant_unique_r
        from all_data
        group by 1,2)
            """)

    logger.info(
        "Blocking: dedup complete -> %s valid candidate pairs ",
        con.sql("SELECT count(*) FROM pairs").limit(1).fetchall()[0][0],
    )
    con.sql("CHECKPOINT;")
    con.sql("""
    SET geometry_always_xy = true;
    CREATE TABLE pairs_minimal AS (
        SELECT
            identifiant_unique_l,
            identifiant_unique_r,
            ST_DISTANCE_Spheroid(
                ST_Point2D(f.latitude,f.longitude),
                ST_Point2D(f2.latitude,f2.longitude)
            ) as geo_distance
        FROM pairs p
        left join features f on p.identifiant_unique_l=f.identifiant_unique
        left join features f2 on p.identifiant_unique_r=f2.identifiant_unique
    )
    """)

    n_pairs = con.sql("SELECT count(*) FROM pairs_minimal").limit(1).fetchall()[0][0]
    if n_pairs == 0:
        # Aucune paire trouvée : on ferme et on signale le cas vide.
        con.close()
        return None, 0

    # On prépare les tables de features pour les jointures finales. On n'emporte
    # QUE les features scalaires dans la jointure : la colonne
    # adresse_clean_vector (1024 x Float32) est exclue et exposée dans une table
    # dédiée `features_vector`, jointe uniquement pour le calcul du cosine.
    column_needed_to_generate_features = [
        "identifiant_unique",
        "nom_clean",
        "ville_clean",
        "siren",
        "siret",
        "telephone",
        "code_commune_insee",
        "code_postal",
        "acteur_type_id",
    ]
    if additional_columns_to_keep is not None:
        column_needed_to_generate_features.extend(additional_columns_to_keep)
        column_needed_to_generate_features = list(
            set(column_needed_to_generate_features)
        )

    con.register(
        "features_full",
        df_features.select(column_needed_to_generate_features),
    )

    # Libère la baseline duckdb : on garde uniquement `pairs_minimal`
    # (l, r, geo_distance) et la table `features_full`, on supprime les tables
    # candidates intermédiaires du blocking pour réduire l'empreinte mémoire.
    for table in [
        "blocking_siren",
        "blocking_departement",
        "blocking_geo",
        "blocking_geo_temp",
        "features",
    ]:
        con.sql(f"DROP TABLE IF EXISTS {table}")
    con.sql("CHECKPOINT;")
    logger.info("Finished blocking (RSS %.0f MB).", _rss_mb())
    return con, n_pairs